# CATE estimation

ATE tells us the average effect across everyone. CATE asks a more useful question for targeting: what's the ad's effect *conditional on* a user's features — does it vary by user, and if so, who benefits most (or is actively hurt)?

Meta-learners covered here, each motivated by a specific flaw in the previous one:
- **S-learner** (foil): one model, treatment as just another feature. Tends to shrink CATE toward zero when the treatment effect is small relative to what covariates explain.
- **T-learner**: two separate models (treated-only, control-only). Fixes S-learner's shrinkage, but its precision is bottlenecked by the smaller group (control, ~150K rows) — the T-learner subtracts one model's raw prediction from another's, so noise from the weaker model passes straight through.
- **X-learner**: designed for unbalanced treatment groups (our 85/15 split). Cross-imputes counterfactuals using the *other* group's model, then fits new models on those imputed effects — this launders the weaker model's noise through a large-sample regression instead of using it raw, and blends the two resulting estimates by propensity score.
- **DR-learner** and **causal forest**: to follow.

**Constraint to hold throughout: never evaluate a CATE model on data used to fit it.** Unlike ATE (a direct calculation, no fitting involved), CATE models are flexible ML models that can overfit — and because no individual ground-truth treatment effect ever exists to check against, an overfit model gives zero warning signal unless checked on a held-out split."

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample, split_train_eval

df = load_sample()
train, eval_ = split_train_eval(df)

print("train:", train.shape, "eval:", eval_.shape)
print("train treatment rate:", train["treatment"].mean())
print("eval treatment rate:", eval_["treatment"].mean())